In [7]:
from client import *
from common_imports import *
from collections.abc import Awaitable, Callable
from agent_framework import (AgentMiddleware,AgentContext,
ChatContext)
import time

In [8]:
async def logging_chat_middleware(
    context: ChatContext,
    call_next: Callable[[], Awaitable[None]],
) -> None:
    """Chat middleware that logs AI interactions."""
    logger.info(f"[💬 Logging][ Chat Middleware] Sending {len(context.messages)} messages to AI")

    await call_next()

    logger.info("[💬 Logging][ Chat Middleware] AI response received")


In [9]:
async def timing_agent_middleware(
    context: AgentContext,
    call_next: Callable[[], Awaitable[None]],
) -> None:
    """Agent middleware that logs execution timing."""
    start = time.perf_counter()
    logger.info("[⏲️ Timing][ Agent Middleware] Starting agent execution")

    await call_next()

    elapsed = time.perf_counter() - start
    logger.info(f"[⏲️ Timing][ Agent Middleware] Execution completed in {elapsed:.2f}s")

In [10]:
def get_weather(
    city: Annotated[str, Field(description="The city to get the weather for.")],
) -> dict:
    """Returns weather data for a given city, a dictionary with temperature and description."""
    logger.info(f"Getting weather for {city}")
    if random.random() < 0.05:
        return {
            "temperature": 72,
            "description": "Sunny",
        }
    else:
        return {
            "temperature": 60,
            "description": "Rainy",
        }


def get_activities(
    city: Annotated[str, Field(description="The city to get activities for.")],
    date: Annotated[str, Field(description="The date to get activities for in format YYYY-MM-DD.")],
) -> list[dict]:
    """Returns a list of activities for a given city and date."""
    logger.info(f"Getting activities for {city} on {date}")
    return [
        {"name": "Hiking", "location": city},
        {"name": "Beach", "location": city},
        {"name": "Museum", "location": city},
    ]


def get_current_date() -> str:
    """Gets the current date from the system and returns as a string in format YYYY-MM-DD."""
    logger.info("Getting current date")
    return datetime.now().strftime("%Y-%m-%d")


In [11]:
agent = Agent(
    client=client,
    instructions=(
        "You help users plan their weekends and choose the best activities for the given weather. "
        "If an activity would be unpleasant in weather, don't suggest it. Include date of the weekend in response."
    ),
    tools=[get_weather, get_activities, get_current_date],
    middleware=[
        # Agent-level middleware applied to ALL runs
        
        logging_chat_middleware,
        timing_agent_middleware,
    ],
)

In [12]:
response = await agent.run("hii what can I do this weekend in San Francisco?")
print(response)

[⏲️ Timing][ Agent Middleware] Starting agent execution
[💬 Logging][ Chat Middleware] Sending 1 messages to AI
[💬 Logging][ Chat Middleware] AI response received
Getting current date
[💬 Logging][ Chat Middleware] Sending 1 messages to AI
[💬 Logging][ Chat Middleware] AI response received
Getting weather for San Francisco
Getting activities for San Francisco on 2026-07-11
Getting activities for San Francisco on 2026-07-12
[💬 Logging][ Chat Middleware] Sending 1 messages to AI
[💬 Logging][ Chat Middleware] AI response received
[⏲️ Timing][ Agent Middleware] Execution completed in 6.77s


This weekend in San Francisco (July 11th and 12th, 2026), the weather is expected to be rainy with a temperature of around 60°F. Since outdoor activities might not be enjoyable in the rain, here are some recommendations:

### Suggested Activities:
1. **Visit a Museum**:
   - Spend time indoors exploring engaging exhibits in San Francisco's museums, like the de Young or Exploratorium.

### Not Recommended:
- **Hiking** and **Beach trips**: These outdoor activities aren't ideal due to the rainy weather.

Stay dry, and enjoy your weekend!
